<a href="https://colab.research.google.com/github/Sid-istic/GNSS_Predictions/blob/main/ML_Model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.svm import SVR
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
train_df = pd.read_csv('train_df.csv')
test_df = pd.read_csv('test_df.csv')
train_df.head()
test_df.head()

,utc_time,x_error(m),y_error(m),z_error(m),satclockerror(m)
0,2025-09-08 14:01:00,0.651484,0.799761,0.851397,0.363912
1,2025-09-08 15:00:00,0.685426,0.678479,0.926463,0.285521
2,2025-09-08 15:00:00,0.685426,0.678479,0.926463,0.285521
3,2025-09-08 16:00:00,0.698550,0.490982,0.915122,0.236231
4,2025-09-08 16:00:00,0.698550,0.490982,0.915122,0.236231


In [ ]:
seconds_in_day = 24 * 60 * 60
train_df["utc_time"] = pd.to_datetime(train_df["utc_time"])
t = train_df["utc_time"].dt.hour * 3600 + train_df["utc_time"].dt.minute * 60 + train_df["utc_time"].dt.second

train_df["time_sin"] = np.sin(2 * np.pi * t / seconds_in_day)
train_df["time_cos"] = np.cos(2 * np.pi * t / seconds_in_day)

In [ ]:
seconds_in_day = 24 * 60 * 60
test_df["utc_time"] = pd.to_datetime(test_df["utc_time"])
t_test = test_df["utc_time"].dt.hour * 3600 + test_df["utc_time"].dt.minute * 60 + test_df["utc_time"].dt.second

test_df["time_sin"] = np.sin(2 * np.pi * t_test / seconds_in_day)
test_df["time_cos"] = np.cos(2 * np.pi * t_test / seconds_in_day)

In [ ]:
LAGS = 16

for lag in range(1, LAGS + 1):
    test_df[f"satclock_lag_{lag}"] = test_df["satclockerror(m)"].shift(lag)
    test_df[f"x_lag_{lag}"] = test_df["x_error(m)"].shift(lag)
    test_df[f"y_lag_{lag}"] = test_df["y_error(m)"].shift(lag)
    test_df[f"z_lag_{lag}"] = test_df["z_error(m)"].shift(lag)

test_df = test_df.dropna()

In [ ]:
LAGS = 16

for lag in range(1, LAGS + 1):
    train_df[f"satclock_lag_{lag}"] = train_df["satclockerror(m)"].shift(lag)
    train_df[f"x_lag_{lag}"] = train_df["x_error(m)"].shift(lag)
    train_df[f"y_lag_{lag}"] = train_df["y_error(m)"].shift(lag)
    train_df[f"z_lag_{lag}"] = train_df["z_error(m)"].shift(lag)

train_df = train_df.dropna()

In [ ]:
train_df.head()

,utc_time,x_error(m),y_error(m),z_error(m),satclockerror(m),time_sin,time_cos,satclock_lag_1,x_lag_1,y_lag_1,...,y_lag_14,z_lag_14,satclock_lag_15,x_lag_15,y_lag_15,z_lag_15,satclock_lag_16,x_lag_16,y_lag_16,z_lag_16
16,2025-09-02 18:00:00,0.760289,0.672068,0.884481,0.185678,-1.000000,-1.836970e-16,0.301709,0.791871,0.825547,...,0.715996,0.838901,0.237027,0.615697,0.859888,0.725431,0.237027,0.615697,0.859888,0.725431
17,2025-09-02 18:00:00,0.760289,0.672068,0.884481,0.185678,-1.000000,-1.836970e-16,0.185678,0.760289,0.672068,...,0.715996,0.838901,0.247677,0.696596,0.715996,0.838901,0.237027,0.615697,0.859888,0.725431
18,2025-09-02 19:00:00,0.641820,0.552710,0.860220,0.180573,-0.965926,2.588190e-01,0.185678,0.760289,0.672068,...,0.603773,0.886542,0.247677,0.696596,0.715996,0.838901,0.247677,0.696596,0.715996,0.838901
19,2025-09-02 19:00:00,0.641820,0.552710,0.860220,0.180573,-0.965926,2.588190e-01,0.180573,0.641820,0.552710,...,0.603773,0.886542,0.332290,0.709823,0.603773,0.886542,0.247677,0.696596,0.715996,0.838901
20,2025-09-02 20:00:00,0.611601,0.494886,0.723447,0.523074,-0.866025,5.000000e-01,0.180573,0.641820,0.552710,...,0.408271,0.883684,0.332290,0.709823,0.603773,0.886542,0.332290,0.709823,0.603773,0.886542


**Linear Regression**

In [ ]:
FEATURES = (
    ["x_error(m)", "y_error(m)", "z_error(m)", "time_sin", "time_cos"] +
    [f"satclock_lag_{i}" for i in range(1, LAGS + 1)] +
    [f"x_lag_{i}" for i in range(1, LAGS + 1)] +
    [f"y_lag_{i}" for i in range(1, LAGS + 1)] +
    [f"z_lag_{i}" for i in range(1, LAGS + 1)]
)

# Define the number of horizons for multi-horizon forecasting
HORIZONS = 3

# Create multi-horizon target variables
y_train_multi = []
y_test_multi = []

for h in range(1, HORIZONS + 1):
    train_df[f"satclockerror_h{h}"] = train_df["satclockerror(m)"].shift(-h)
    test_df[f"satclockerror_h{h}"] = test_df["satclockerror(m)"].shift(-h)
    y_train_multi.append(train_df[f"satclockerror_h{h}"])
    y_test_multi.append(test_df[f"satclockerror_h{h}"])

# Drop rows with NaN values resulting from shifting for horizons
train_df_multi = train_df.dropna()
test_df_multi = test_df.dropna()

# Convert lists of Series to 2D NumPy arrays
y_train = np.array(y_train_multi).T
y_test = np.array(y_test_multi).T

# Filter X_train_ml and X_test_ml to match the dropped rows
X_train_ml = train_df_multi[FEATURES]
X_test_ml  = test_df_multi[FEATURES]

# Ensure y_train and y_test have the same number of rows as X_train_ml and X_test_ml
# This slicing ensures alignment after dropping NaNs
y_train = y_train[:len(X_train_ml)]
y_test = y_test[:len(X_test_ml)]

# The original single-horizon target for other models (if still needed)
y_train_ml = train_df_multi["satclockerror(m)"] # This will be aligned with the multi-horizon data
y_test_ml  = test_df_multi["satclockerror(m)"]  # This will be aligned with the multi-horizon data

lr = LinearRegression()
lr.fit(X_train_ml, y_train_ml)

pred_lr = lr.predict(X_test_ml)

mae_lr = mean_absolute_error(y_test_ml, pred_lr)
mse_lr = mean_squared_error(y_test_ml, pred_lr)
rmse_lr = mse_lr**0.5 # Calculate RMSE by taking the square root of MSE

print("Linear Regression \u2192 MAE:", mae_lr, "| RMSE:", rmse_lr)

Linear Regression → MAE: 0.030573897858900613 | RMSE: 0.039277573697395964


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

lr_results = {}

# HORIZONS is already defined in the previous cell
for h in range(HORIZONS):

    # y_train and y_test are now 2D arrays, so we can index them by column
    y_train_h = y_train[:, h]
    y_test_h = y_test[:, h]

    model = LinearRegression()
    model.fit(X_train_ml, y_train_h)

    preds = model.predict(X_test_ml)

    mae = mean_absolute_error(y_test_h, preds)
    rmse = np.sqrt(mean_squared_error(y_test_h, preds))

    lr_results[f"H{h+1}"] = {"MAE": mae, "RMSE": rmse}

    print(f"Horizon {h+1}")
    print("MAE:", mae)
    print("RMSE:", rmse)

Horizon 1
MAE: 0.03622542063941809
RMSE: 0.04385361297735824
Horizon 2
MAE: 0.03615612934468066
RMSE: 0.043670190239267714
Horizon 3
MAE: 0.034396796407899874
RMSE: 0.04262000202274679


**Support Vector Regressor**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

svr_results = {}

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_ml)
X_test_scaled = scaler.transform(X_test_ml)

for h in range(3):

    y_train_h = y_train[:, h]
    y_test_h = y_test[:, h]

    model = SVR(kernel="rbf", C=10, epsilon=0.01)
    model.fit(X_train_scaled, y_train_h)

    preds = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test_h, preds)
    rmse = np.sqrt(mean_squared_error(y_test_h, preds))

    svr_results[f"H{h+1}"] = {"MAE": mae, "RMSE": rmse}

    print(f"Horizon {h+1}")
    print("MAE:", mae)
    print("RMSE:", rmse)

Horizon 1
MAE: 0.04444803481916215
RMSE: 0.05831751251763623
Horizon 2
MAE: 0.0598015636368952
RMSE: 0.07461943383290255
Horizon 3
MAE: 0.05877032427594688
RMSE: 0.07259795852737184


**RandomForest Regressor**

In [ ]:
FEATURES = [
    "x_error(m)", "y_error(m)", "z_error(m)",
    "time_sin", "time_cos"
] + [f"satclock_lag_{i}" for i in range(1, 17)]

TARGET = "satclockerror(m)"

X_train_ml = train_df[FEATURES]
y_train_ml = train_df[TARGET]

X_test_ml  = test_df[FEATURES]
y_test_ml  = test_df[TARGET]

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train_ml, y_train_ml)

preds_rf = rf.predict(X_test_ml)

mae_rf = mean_absolute_error(y_test_ml, preds_rf)
print("RF MAE:", mae_rf)

RF MAE: 0.013104857587325172


In [ ]:
print(max(train_df["satclockerror(m)"]))
print(min(train_df["satclockerror(m)"]))

1.0
0.0
